# Module 09 — Exceptions, `with` and logging

Module 08 left two questions open: what `with` actually is, and what to do about a
missing file other than checking for it first. Both are here, along with the
difference that will cost you the most habits — Python has no checked exceptions.

## 1. There is nothing in the signature

```java
static String read(Path p) throws IOException     // Java: declared, and the caller must deal with it
```

```python
def read(path: Path) -> str: ...                  # Python: says nothing about what can go wrong
```

No `throws`, no compiler counting the cases, no distinction between checked and
unchecked. Any function can raise anything, and nothing at the call site is required
to mention it.

What replaces the compiler: the documentation, the type checker for the return value,
and a habit — catch the exception you can actually do something about, as close to
the operation as you can, and let the rest travel. `exercises/thinking.md` asks what
that trade is worth.

## 2. Catch what you meant

An `except` clause catches its class **and everything below it**. So the clause you
write decides how much you have taken responsibility for.

In [ ]:
def convert(raw):
    try:
        return float(raw)
    except ValueError:  # exactly the failure this line can produce
        return None


print(convert("21.7"), convert("n/a"), convert(""))
print(FileNotFoundError.__mro__[:4])  # the parent chain: an except for OSError catches this too

`except Exception` catches nearly everything. A **bare** `except:` catches more than
that, and the difference is not academic:

In [ ]:
print(issubclass(ValueError, Exception))
print(issubclass(KeyboardInterrupt, Exception))  # False
print(issubclass(SystemExit, Exception))  # False
print(KeyboardInterrupt.__mro__[:3])

`KeyboardInterrupt` and `SystemExit` inherit from `BaseException`, not from
`Exception`. They are not errors in the program — they are the interpreter being told
to stop. A bare `except:` catches them too, which means a loop with one in it cannot
be stopped with Ctrl-C and a `sys.exit()` inside it does not exit.

So: **`except Exception` when you genuinely mean everything, and never the bare
form.** Better still, name the class. `ruff`'s `E722` exists for exactly this line.

## 3. `else` and `finally`

Java has `finally`. `else` — the block that runs only if the `try` raised nothing —
has no counterpart, and its job is to keep the `try` down to the line that can
actually fail.

In [ ]:
def parse(raw):
    order = []
    try:
        order.append("try")
        value = float(raw)
    except ValueError:
        order.append("except")
        value = None
    else:
        order.append("else")  # only when nothing raised
    finally:
        order.append("finally")  # on every way out of the try
    return value, order


print(parse("21.7"))
print(parse("n/a"))

Why that matters: everything you put in the `try` is covered by the `except`. Put
three more lines in and a `ValueError` from any of them is caught by a clause that
was written for the conversion. The `else` block is not covered — which is where the
rest of the work belongs.

`finally` runs on the way out of the block whichever way you leave — falling off the
end, an exception, or a `return`. The last one is the interesting case. Predict it
before you run it.

In [ ]:
def sneaky():
    try:
        return "from try"
    finally:
        return "from finally"


assert sneaky() == ...

A `return` in `finally` replaces the one from the `try` — and would swallow an
exception on its way past. It is legal, it is almost never what you want, and it is
worth having seen once so you recognise it in someone else's code.

Java behaves the same way here, which makes this one of the places where knowing the
Java rule is enough.

## 4. Forgiveness, not permission

Module 08 checked `path.exists()` and then opened the file. Between those two lines
the file can be deleted, renamed, or have its permissions changed — by another
process, or by the user. The check does not make the open safe; it only makes the
failure less frequent and therefore harder to reproduce.

In [ ]:
from pathlib import Path

missing = Path("data") / "nope.csv"

# Look before you leap -- and the gap between the two lines is real
if missing.exists():
    text = missing.read_text(encoding="utf-8")
else:
    text = ""

# Ask forgiveness -- one operation, one outcome
try:
    text = missing.read_text(encoding="utf-8")
except FileNotFoundError:
    text = ""

print(repr(text))

The names for the two styles are LBYL (look before you leap) and EAFP (easier to ask
forgiveness than permission). Python leans on the second one throughout, and it is
not only about the race: the check duplicates the condition the operation is going to
test anyway, and the two copies can disagree.

Where LBYL is still right: when the check is cheap, has no race, and the answer is
part of the logic rather than an error — `if key in config:` reads better than
catching `KeyError` when both branches are ordinary.

## 5. Raising, and what Python remembers

`raise` takes an instance. Raise inside an `except` block and the original is kept
and printed with the new one.

In [ ]:
def parse(raw):
    try:
        return float(raw)
    except ValueError:
        # no `from` here -- see what Python remembers anyway
        raise ValueError(f"not a reading: {raw!r}")


try:
    parse("n/a")
except ValueError as err:
    print("raised:  ", err)
    print("context: ", type(err.__context__).__name__)  # kept automatically
    print("cause:   ", err.__cause__)  # None -- nothing said it was deliberate

Printed as a traceback, that reads *During handling of the above exception, another
exception occurred*. The wording is deliberately non-committal: Python knows the
first exception was still being handled when the second was raised, and nothing more.

`from` changes the wording and the meaning: *The above exception was the direct cause
of the following exception.*

In [ ]:
def parse(raw):
    try:
        return float(raw)
    except ValueError as err:
        raise ValueError(f"not a reading: {raw!r}") from err


try:
    parse("n/a")
except ValueError as err:
    print("cause:", type(err.__cause__).__name__)

Use `from err` when you are translating an exception into one that fits your own
layer — which is most of the time. `from None` suppresses the original entirely, and
is for the case where the internal failure would only confuse the reader.

An exception class of your own is one line:

In [ ]:
class ParseError(Exception):
    """Raised when a line of the log cannot be read."""


try:
    raise ParseError("line 4: value is empty")
except ParseError as err:  # your class, caught like any other
    print(type(err).__name__, "-", err)

print(ParseError.__mro__[:3])

A name, a parent, and a docstring. That is the whole of what you need from classes
until module 11 — and inheriting from `Exception` rather than `BaseException` is what
puts your class where `except Exception` can see it.

Define one when a caller might reasonably want to catch *your* failure and not
everything else that happens to be a `ValueError`.

## 6. `with` is a protocol

Java's try-with-resources works on anything implementing `AutoCloseable`, and calls
`close()`. Python's `with` works on anything implementing `__enter__` and `__exit__`,
and the object decides what those do — closing is only the most common case.

In [ ]:
class Section:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"-> {self.name}")
        return self  # whatever this returns is what `as` binds

    def __exit__(self, exc_type, exc, traceback):
        print(f"<- {self.name} (raised: {exc_type.__name__ if exc_type else 'nothing'})")
        return False  # False: the exception carries on


with Section("reading") as section:
    print("   working in", section.name)

try:
    with Section("parsing"):
        raise ValueError("bad line")
except ValueError:
    print("   the exception reached the caller")

`__exit__` is called on the way out either way, which is what makes it the right place
for a release. Its three arguments are `None, None, None` when the block ended
normally, and describe the exception when it did not.

**A truthy return from `__exit__` swallows the exception.** That, and `__enter__`
deciding what `as` binds, are the two parts of the protocol try-with-resources has no
answer for — there the resource simply is the variable. Swallowing is how
`contextlib.suppress` works.

In [ ]:
import contextlib

with contextlib.suppress(FileNotFoundError):
    open("does-not-exist.csv", encoding="utf-8")

print("still here -- suppress swallowed it")

Writing one as a class is fine; writing one as a function is shorter. Everything
before the `yield` is `__enter__`, everything after is `__exit__`, and the `try` /
`finally` is what makes the second half run when the block raised.

In [ ]:
import contextlib
import time


@contextlib.contextmanager
def timed(label):
    start = time.perf_counter()
    try:
        yield  # the with-block runs here
    finally:
        print(f"{label}: {(time.perf_counter() - start) * 1000:.1f} ms")


with timed("counting"):
    total = sum(range(200_000))

print(total)

Two more things `with` does that are worth knowing: several managers in one statement
(`with open(a) as x, open(b) as y:`), and the fact that a file object is a context
manager, which is why module 08 could use one before any of this was explained.

## 7. Reading a traceback

Python prints the call chain **oldest first**, so the line that raised is at the
bottom, immediately above the exception. Java prints the innermost frame at the top.

In [ ]:
def read_value(raw):
    return float(raw)


def read_row(row):
    return read_value(row["value"])


read_row({"value": "n/a"})

Bottom line: the exception type and its message. Just above: the line that raised.
Above that: who called it, and who called that. **Read the last line first**, then walk
up until you reach a frame in your own code.

## 8. `logging`

`print` writes to standard output, has no level, no timestamp, no source, and cannot
be turned down without editing the code. That is fine while you are watching. It is
not fine for anything that runs unattended, which is everything from module 20 on.

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,  # INFO and above are emitted; DEBUG is dropped
    format="%(levelname)s %(name)s: %(message)s",
    force=True,  # replace any handler a previous cell installed
)

log = logging.getLogger("sensors")  # one logger per module: __name__ is the usual argument

log.debug("dropped: below the level")
log.info("read %d rows", 5)  # arguments, not an f-string -- see below
log.warning("%s above limit: %.1f", "TH-04", 91.0)

try:
    float("n/a")
except ValueError:
    log.exception("could not parse")  # ERROR plus the traceback, inside an except block

Three things to take from that cell:

- **The level decides what is emitted**, and it is set in one place rather than at
  every call. `DEBUG` `INFO` `WARNING` `ERROR` `CRITICAL`; the default for the root
  logger is `WARNING`, which is why an `info` call in a fresh program appears to do
  nothing.
- **`log.info("read %d rows", 5)`, not `log.info(f"read {5} rows")`.** The arguments
  are only formatted if the message is actually emitted, so a `debug` call in a hot
  loop costs almost nothing when debugging is off. This is the one place where the
  `%` form of module 07 is still the recommended one.
- **`log.exception` belongs inside an `except` block** and adds the traceback by
  itself.

`logging.getLogger(__name__)` at the top of a module gives every message the module
it came from, and lets the level be set per package. That is the whole of the setup
you need until module 20.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

`data/readings.csv` has two rows that cannot be parsed. Several of the exercises are
about deciding what a program should do with them.

Module 10 is the last of Part 2: modules, packages, `__main__`, and what `uv` does
with a project.